<a href="https://colab.research.google.com/github/PhucPower300121/FLUX-Jupyter/blob/main/flux2_klein_i2i_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### FLUX.2 [klein] 4B Image Edit — Colab (ComfyUI + GGUF Q4)

Credit: [ComfyUI](https://github.com/comfyanonymous/ComfyUI), [ComfyUI-GGUF (city96)](https://github.com/city96/ComfyUI-GGUF), [FLUX.2-klein-4B-GGUF (unsloth)](https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF)

In [ ]:
#@title Install ComfyUI + ComfyUI-GGUF + download model
%cd /content/
!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI

PIN_COMMIT = ""  # để trống = bản mới nhất, hoặc điền hash commit cụ thể để cố định version
if PIN_COMMIT:
    !git fetch --all -q
    !git reset --hard {PIN_COMMIT}

!pip install -q -r requirements.txt

# Cài custom node GGUF
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF comfyui_gguf
!pip install -q -r comfyui_gguf/requirements.txt

%cd /content/ComfyUI
import os
os.makedirs("/content/ComfyUI/models/unet", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)

!apt -y install -qq aria2

# GGUF Q4_K_S, mmap load -> không đè RAM system trước khi qua GPU
unet_path = '/content/ComfyUI/models/unet/flux-2-klein-4b-Q4_K_S.gguf'
vae_path = '/content/ComfyUI/models/vae/flux2-vae.safetensors'
text_encoder_path = '/content/ComfyUI/models/text_encoders/qwen_3_4b_fp4_flux2.safetensors'

if not os.path.exists(unet_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_K_S.gguf -d /content/ComfyUI/models/unet -o flux-2-klein-4b-Q4_K_S.gguf
if not os.path.exists(vae_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/vae/flux2-vae.safetensors -d /content/ComfyUI/models/vae -o flux2-vae.safetensors
if not os.path.exists(text_encoder_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors -d /content/ComfyUI/models/text_encoders -o qwen_3_4b_fp4_flux2.safetensors

from IPython.display import clear_output
clear_output()

paths = [unet_path, vae_path, text_encoder_path]
missing = [p for p in paths if not os.path.exists(p)]
if missing:
    print("\033[91mMISSING FILE (Rerun this cell):\033[0m", missing)
else:
    print("\033[92mInstall + download model complete.\033[0m")

In [ ]:
#@title Load model to RAM (GGUF, mmap -- low RAM)
%cd /content/ComfyUI

import random, torch, numpy as np
from PIL import Image
from nodes import NODE_CLASS_MAPPINGS
import nodes

# Nạp toàn bộ node built-in (comfy_extras) + custom node (GGUF...) qua cơ chế chính thức của ComfyUI
from nodes import init_extra_nodes
await init_extra_nodes()

UnetLoaderGGUF = NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
LoadImage = nodes.LoadImage()

with torch.inference_mode():
    clip = CLIPLoader.load_clip("qwen_3_4b_fp4_flux2.safetensors", "flux2")[0]
    unet = UnetLoaderGGUF.load_unet("flux-2-klein-4b-Q4_K_S.gguf")[0]
    vae = VAELoader.load_vae("flux2-vae.safetensors")[0]

print("Model load complete.")

In [ ]:
#@title Upload photos (Support to select multiple photos)
from google.colab import files
uploaded = files.upload()
import shutil, os
init_image_names = []
for fname in uploaded.keys():
    dst = os.path.join("/content/ComfyUI/input", fname)
    shutil.copy(fname, dst)
    init_image_names.append(fname)
print("Uploaded:", init_image_names)

In [ ]:
#@title run Image Edit
positive_prompt = "for a hand caressing the character's head"  #@param {type:"string"}
negative_prompt = ""  #@param {type:"string"}
steps = 4  #@param {type:"slider", min:4, max:30, step:1}
cfg = 1.0  #@param {type:"number"}
sampler_name = "euler"  #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde"]
scheduler = "simple"  #@param ["simple", "normal", "karras"]
seed = 0  #@param {type:"integer"}

# 4 step + cfg 1.0 la config distilled mac dinh cua Klein.
# Neu dung ban base (khong distilled), tang steps ~20-30 va cfg ~3.5-4.0.

with torch.inference_mode():
    if seed == 0:
        seed = random.randint(0, 18446744073709551615)
    print("Seed:", seed)

    import comfy.utils

    def load_and_scale(fname, target_px=1024*1024):
        img = LoadImage.load_image(fname)[0]
        s = img.movedim(-1, 1)
        total_px = s.shape[3] * s.shape[2]
        scale_by = (target_px / total_px) ** 0.5
        new_w = max(32, round(s.shape[3] * scale_by / 32) * 32)
        new_h = max(32, round(s.shape[2] * scale_by / 32) * 32)
        s = comfy.utils.common_upscale(s, new_w, new_h, "lanczos", "disabled")
        return s.movedim(1, -1)

    # Moi anh input duoc scale rieng ve ~1MP roi encode thanh 1 reference latent.
    # FLUX.2 nhan nhieu reference_latents cung luc de tong hop vao 1 output.
    ref_images = [load_and_scale(fname) for fname in init_image_names]
    ref_latents_list = [VAEEncode.encode(vae, img)[0]["samples"] for img in ref_images]

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    positive = [[t[0], {**t[1], "reference_latents": (t[1].get("reference_latents", []) + ref_latents_list)}] for t in positive]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]

    # Canvas output lay kich thuoc theo anh dau tien
    lat_h, lat_w = ref_images[0].shape[1] // 8, ref_images[0].shape[2] // 8
    empty_latent = {"samples": torch.zeros([1, 16, lat_h, lat_w])}

    samples = KSampler.sample(
        unet, seed, steps, cfg, sampler_name, scheduler,
        positive, negative, empty_latent, denoise=1.0
    )[0]

    decoded = VAEDecode.decode(vae, samples)[0].detach()
    out_img = Image.fromarray(np.array(decoded * 255, dtype=np.uint8)[0])
    out_img.save("/content/output.png")

out_img

In [ ]:
#@title Download output
from google.colab import files
files.download("/content/output.png")